# GIAI ĐOẠN 2: DATA CLEANING & PREPROCESSING
Mục tiêu:: Biến dữ liệu thô từ nhiều nguồn khác nhau thành dữ liệu sạch sẵn sàng cho phân tích và huấn luyện mô hình
**Quy trình xử lý**
1. **Ingestion & Merging:** Gộp dữ liệu đa luồng 
2. **Data Profiling:** Soi dữ liệu
3. **Standardization:** Chuẩn hoá định dạng văn bản
3. **Extraction:** Trích xuất đặc trưng(Quận/Huyện,Giá,Diện tích, Tiện Ích)
4. **Cleaning Logic:** Xử lý trùng lặp, dữ liệu khuyết và Outliers
5. **Enrichment:** Phân vùng địa lý(Zoning)

In [1]:
import pandas as pd
import numpy as np
import re # Thư viện xử lý Regular Expression (Regex)
from datetime import datetime

# Cài đặt hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
import warnings
warnings.filterwarnings('ignore')

INPUT_FILES = [
    '../data/raw/homedy_raw.csv',
    '../data/raw/phongtro123_raw.csv',
    '../data/raw/tromoi_raw.csv'
]

OUTPUT_FILE = '../data/processed/clean_data_file.csv'

print('Đã import thư viện và cấu hình xong')

Đã import thư viện và cấu hình xong


## Bước 1: Data Ingestion & Merging (Gộp dữ liệu)
Chúng ta load tất cả các file csv thô, đổi tên cột về chuấn tiếng anh(optional), và gộp lại

In [2]:
def load_and_merge(file_list):
    all_dfs = []
    for f in file_list:
        try:
            # load dữ liệu
            df = pd.read_csv(f)
            # kiểm tra nhanh file có đủ cột không
            required_cols = ['title', 'price', 'area', 'address', 'description']
            if not all(col in df.columns for col in required_cols):
                print(f"Cảnh báo: File {f} thiếu cột chuẩn. Bỏ qua.")
                continue
            
            all_dfs.append(df)
            print(f"-> Đã load {f}: {len(df)} dòng")
        except Exception as e:
            print(f"Lỗi đọc file {f}: {e}")
    
    if not all_dfs: return None
    
    # gộp
    df_master = pd.concat(all_dfs, ignore_index= True)
    
    # Bỏ cột index thừa (vì pandas khi read_csv sẽ tự tạo index mới)
    if 'index' in df_master.columns:
        df_master = df_master.drop(columns=['index'])
    
    return df_master

# chạy load 
df = load_and_merge(INPUT_FILES)
print(f"TỔNG DỮ LIỆU: {len(df)} dòng")
display(df.head(2))

-> Đã load ../data/raw/homedy_raw.csv: 1944 dòng
-> Đã load ../data/raw/phongtro123_raw.csv: 12000 dòng
-> Đã load ../data/raw/tromoi_raw.csv: 711 dòng
TỔNG DỮ LIỆU: 14655 dòng


,url,title,price,area,address,description,source,posted_time,owner_name,phone
0,https://homedy.com/cho-thue-nha-tro-phong-tro-quan-tan-binh-tp-ho-chi-minh/chdv-30m2-gac-cao-cua...,"CHDV 30m2 gác cao cửa sổ lớn đường Giải Phóng Tân Bình chỉ 5,5 triệu","5,5 Triệu/tháng",30 m2,"Đường Giải Phóng, Phường 4, Quận Tân Bình, TP Hồ Chí Minh",CHÍNH CHỦ CHO THUÊ CHDV ĐƯỜNG GIẢI PHÓNG TÂN BÌNH CÓ GÁC CHỈ 5 TRIỆU 500k - Địa chỉ: Đường Giải...,homedy.com,NaN,NaN,NaN
1,https://homedy.com/cho-thue-nha-tro-phong-tro-quan-tan-binh-tp-ho-chi-minh/con-duy-nhat-1-duong-...,"Còn duy nhất 1 phòng cho thuê_ Đường Hoàng Văn Thụ, P.2, Q. Tân Bình, Hồ Chí Minh","3,5 Triệu/tháng",26 m2,"Đường Hoàng Văn Thụ, Phường 2, Quận Tân Bình, TP Hồ Chí Minh","Khu dân cư yên tĩnh, an ninh, ngay vòng xoay Lăng Cha Cả. Thuận tiện di chuyển muôn nơi (3-5 phú...",homedy.com,NaN,NaN,NaN


## Bước 2: Data profiling(Soi dữ liệu)

Mục tiêu: Tìm ra format lạ để viết hàm xử lý cho đúng<div>

In [8]:
def inspect_data_by_source(df, col_name):
    print(f"\n=== KIỂM TRA CỘT: {col_name.upper()} ===")
    
    sources = df['source'].unique()
    for source in sources:
        print(f"\nNGUỒN: {source}")
        # 1. Lọc lấy dữ liệu của riêng nguồn này
        df_source = df[df['source']== source]
        
        # 2. Tìm các giá trị khả nghi chứa chữ cái
        weird_values = df_source[df_source[col_name].astype(str).str.contains(r'[a-zA-Z]', na=False)]
    
        # 3. Lấy danh sách các kiểu viết lạ(Unique)
        unique_weird = weird_values[col_name].unique()
        
        # 4. In ra kế quả (Chỉ in tối đa 10 mẫu)
        if len(unique_weird) > 0:
            print(f"   - Phát hiện {len(weird_values)} dòng chứa chữ.")
            print(f"   - Các mẫu format lạ (Top 10):")
            print(f"     {unique_weird[:10]}") # In 10 mẫu đầu tiên của nguồn này
        else:
            print(f"   - Sạch (Chỉ toàn số hoặc NaN).")
    
        # 5. Soi chi tiết từng độ dài (Step-by-step scan)
        s_values = df_source[col_name].dropna().astype(str)
        
        if len(s_values) > 0:
            lengths = s_values.str.len()
            min_len = int(lengths.min())
            max_len = int(lengths.max())
            
            print(f"   - Độ dài chuỗi: Min={min_len}, Max={max_len}")
            
            # Nếu khoảng cách Min-Max quá lớn (ví dụ > 50), chỉ in những độ dài có dữ liệu để tránh spam
            # Nếu khoảng cách nhỏ (ví dụ 12-20), in full như ý bạn
            range_len = range(min_len, max_len + 1)
            
            # Chỉ in tối đa 30 dòng để tránh treo máy nếu có outlier (VD: Min=5, Max=1000)
            if max_len - min_len > 30:
                print(f"     (Khoảng cách quá lớn, chỉ in các độ dài có xuất hiện dữ liệu...)")
                scan_list = sorted(lengths.unique()) # Chỉ lấy các độ dài có thật
            else:
                scan_list = range_len # Quét full từ Min đến Max (kể cả rỗng)

            for l in scan_list:
                # Tìm label (Min/Max)
                label = f"Mẫu {l}"
                if l == min_len: label += "(Min)"
                elif l == max_len: label += "(Max)"
                
                # Lấy 3 mẫu unique
                samples = s_values[lengths == l].unique()[:3].tolist()
                
                # In ra (Chỉ in nếu có dữ liệu hoặc nếu đang ở chế độ quét full)
                if len(samples) > 0:
                    print(f"     + {label:<15}: {samples}")
                elif max_len - min_len <= 30: # Nếu list rỗng nhưng nằm trong range nhỏ thì vẫn in []
                     print(f"     + {label:<15}: []")

        else:
             print("   - (Không có dữ liệu)")
        
        
             
inspect_data_by_source(df, 'price')
inspect_data_by_source(df, 'area')        


=== KIỂM TRA CỘT: PRICE ===

NGUỒN: homedy.com
   - Phát hiện 1944 dòng chứa chữ.
   - Các mẫu format lạ (Top 10):
     ['5,5 Triệu/tháng' '3,5 Triệu/tháng' '4,5 Triệu/tháng' '1,4 Triệu/tháng'
 '0,75 Triệu/tháng' '1,2 Triệu/tháng' '1 Triệu/tháng'
 '3,5 - 5 Triệu/tháng' '3,8 - 5 Triệu/tháng' '4,8 Triệu/tháng']
   - Độ dài chuỗi: Min=4, Max=23
     + Mẫu 4(Min)     : ['7 Tỷ']
     + Mẫu 5          : []
     + Mẫu 6          : []
     + Mẫu 7          : ['10,3 Tỷ']
     + Mẫu 8          : []
     + Mẫu 9          : []
     + Mẫu 10         : ['Thỏa thuận']
     + Mẫu 11         : []
     + Mẫu 12         : []
     + Mẫu 13         : ['1 Triệu/tháng', '4 Triệu/tháng', '3 Triệu/tháng']
     + Mẫu 14         : ['15 Triệu/tháng', '10 Triệu/tháng', '12 Triệu/tháng']
     + Mẫu 15         : ['5,5 Triệu/tháng', '3,5 Triệu/tháng', '4,5 Triệu/tháng']
     + Mẫu 16         : ['0,75 Triệu/tháng', '0,95 Triệu/tháng', '0,65 Triệu/tháng']
     + Mẫu 17         : ['6 - 9 Triệu/tháng', '3 - 5 Triệu/thán

**3 Phát hiện quan trọng từ dữ liệu:**<div>
1. Xung đột dấu thập phân:
- homedy: 5,5 Triệu
- phongtro123: 9.5 triệu
- phongtro123: 5.000.000 đồng 
2. Xuất hiện khoảng giá trị 7 - 15 m2, 6 - 9 Triệu/tháng 
3. Dữ liệu rỗng: m2 mà không có số phía trước
4. xuất hiện mẫu lạ 'Thoả thuận'
<div>


## BƯỚC 3: STANDARDIZATION


**Chiến thuật xử lý:**<div>
Bước 1 (Pre-clean): Nếu gặp nguồn homedy, thay thế ngay , thành .

Bước 2 (Phân loại dấu chấm):
Nếu chuỗi chứa "triệu" hoặc "tỷ" -> Dấu . là thập phân (Giữ nguyên)
Nếu chuỗi chứa "đồng" hoặc không có đơn vị -> Dấu . là hàng nghìn (Xóa đi)

Bước 3 (Range): Tách dấu -, tính trung bình cộng.


**Kết quả mong đợi với price**<div>
Input: 5,5 Triệu/tháng      -> Output: 5,500,000<div>
Input: 9.5 triệu/tháng      -> Output: 9,500,000<div>
Input: 5.000.000 đồng/tháng -> Output: 5,000,000<div>
Input: 3,5 - 5 Triệu/tháng  -> Output: 4,250,000<div>
Input: 5 đồng/tháng         -> Output: 5<div>
Input: 0,75 Triệu/tháng     -> Output: 750,000<div>
Input: 7 Tỷ                 -> Output: 7,000,000,000<div>

**Kết quả mong đợi với area**<div>
Input: 25 m2                 -> Output: 25 <div>
Input: 100m²                 -> Output: 100 <div>
Input: 8 - 10 m2             -> Output: 9 

In [17]:
def parse_price(price_raw):
    if pd.isna(price_raw): return np.nan
    s = str(price_raw).lower().strip()
    
    # 0. Check các trường hợp đặc biệt ngay từ đầu
    if 'thỏa thuận' in s or 'thoa thuan' in s: return np.nan
    
    # 1. Xác định "Ngữ cảnh toàn câu" (Global Context)
    # Để biết "3,5 - 5 Triệu" thì số 3,5 kia cũng phải là Triệu
    multiplier = 1
    has_explicit_unit = False # Cờ đánh dấu đã tìm thấy đơn vị to
    
    if 'tỷ' in s or 'ty' in s:
        multiplier = 1_000_000_000
        has_explicit_unit = True
    elif 'triệu' in s or 'tr' in s:
        multiplier = 1_000_000
        has_explicit_unit = True
    elif 'usd' in s:
        multiplier = 25_000
        has_explicit_unit = True
    elif 'ngàn' in s or 'k ' in s or s.endswith('k'): # k phải đứng riêng hoặc cuối câu
        multiplier = 1_000
        has_explicit_unit = True
        
    # 2. Check "Đơn vị nhỏ" (Đồng/VND) -> Để CHẶN heuristic nhân triệu
    is_small_unit = False
    if 'đồng' in s or 'đ/' in s or 'vnd' in s or 'uud' in s: # uud do lỗi font
        is_small_unit = True

    # 3. Chuẩn hóa dấu phẩy (5,5 -> 5.5)
    s = s.replace(',', '.')

    # 4. Hàm con: Tách số từ 1 đoạn text con
    def extract_number(sub_s):
        # Xóa các dấu chấm phân cách hàng nghìn nếu đó là đơn vị nhỏ
        # VD: 5.000.000 đồng -> 5000000
        if is_small_unit or (not has_explicit_unit and sub_s.count('.') >= 2):
            sub_s = sub_s.replace('.', '')
            
        nums = re.findall(r"[-+]?\d*\.\d+|\d+", sub_s)
        if nums:
            val = float(nums[0])
            
            # LOGIC QUYẾT ĐỊNH HỆ SỐ NHÂN (QUAN TRỌNG NHẤT)
            # Nếu đã xác định được Global Unit (Triệu/Tỷ) -> Nhân luôn
            if has_explicit_unit:
                # Nhưng nếu số đã quá lớn (VD: user nhập 3.000.000 triệu) -> thì không nhân nữa
                if val > 1000 and multiplier >= 1_000_000:
                    return val 
                return val * multiplier
            
            # Nếu KHÔNG có đơn vị to, và KHÔNG phải đơn vị nhỏ (đồng)
            # Thì mới áp dụng Heuristic < 1000 => Nhân triệu
            if not is_small_unit and val < 1000:
                return val * 1_000_000
                
            return val
        return None

    # 5. Xử lý Range (3.5 - 5)
    if '-' in s:
        parts = s.split('-')
        if len(parts) >= 2:
            v1 = extract_number(parts[0])
            v2 = extract_number(parts[1])
            
            # Nếu tách được cả 2 thì lấy trung bình
            if v1 is not None and v2 is not None:
                return (v1 + v2) / 2
            # Nếu chỉ tách được 1 số (số kia rác)
            return v1 if v1 is not None else v2

    # 6. Xử lý số đơn
    return extract_number(s)

# --- CHẠY LẠI TEST CASE CỦA BẠN (ĐỂ KIỂM CHỨNG) ---
test_cases = [
    '5,5 Triệu/tháng',      
    '9.5 triệu/tháng',      
    '5.000.000 đồng/tháng', 
    '3,5 - 5 Triệu/tháng',  # Ca khó nhất
    '5 đồng/tháng',         # Ca bị lỗi cũ
    '0,75 Triệu/tháng',     
    '7 Tỷ'                 
]

print("KẾT QUẢ LOGIC:")
for t in test_cases:
    val = parse_price(t)
    print(f"Input: {t:<22} -> Output: {val:,.0f}" if pd.notna(val) else f"Input: {t:<22} -> Output: NaN")
    
def parse_area(area_raw):
    """
    Xử lý: '30m2', '30 - 40m2', '4x15', '5*20'
    """
    if pd.isna(area_raw): return np.nan
    s = str(area_raw).lower().strip()
    
    # 0. Nếu không có số nào -> NaN (VD: chỉ có chữ 'm2')
    if not any(char.isdigit() for char in s):
        return np.nan

    # 1. Dọn dẹp đơn vị và dấu phẩy
    s = s.replace('m²', '').replace('m2', '').replace('s', '') # Có nơi viết 30s (square)
    s = s.replace(',', '.') 

    # 2. Xử lý Kích thước Dài x Rộng (4x15 hoặc 4*15)
    # Logic: Nếu thấy dấu x hoặc *, tách ra và nhân lại
    if 'x' in s or '*' in s:
        # Chuẩn hóa về 'x'
        s_dim = s.replace('*', 'x')
        parts = s_dim.split('x')
        if len(parts) >= 2:
            try:
                # Tìm số trong từng phần (VD: "4m" x "15m")
                v1 = float(re.findall(r"[\d.]+", parts[0])[0])
                v2 = float(re.findall(r"[\d.]+", parts[1])[0])
                return v1 * v2 # Trả về diện tích tính toán
            except:
                pass # Nếu lỗi thì xuống bước dưới xử lý bình thường

    # 3. Xử lý Range (30 - 45) -> Trung bình cộng
    if '-' in s:
        parts = s.split('-')
        if len(parts) >= 2:
            try:
                v1 = float(re.findall(r"[\d.]+", parts[0])[0])
                v2 = float(re.findall(r"[\d.]+", parts[1])[0])
                return (v1 + v2) / 2
            except: 
                pass

    # 4. Tách số đơn thuần
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", s)
    if nums:
        return float(nums[0])
        
    return np.nan

# --- CHẠY THỬ NGHIỆM CASE DIỆN TÍCH ---
test_area_cases = [
    '30 m2',
    '30 - 40 m2',     # Range
    '4x15',           # Dimension (Dài x Rộng)
    '5*20 m2',        # Dimension biến thể
    '15,5 m2',        # Dấu phẩy
    'm2'              # Rác
]

print("\nKIỂM TRA AREA:")
for t in test_area_cases:
    val = parse_area(t)
    print(f"Input: {t:<15} -> Output: {val}")


print(f"Dữ liệu trước khi chuẩn hóa: {df.shape}")

print("⏳ Đang chạy chuẩn hóa cột PRICE...")
# Tạo cột mới 'price', giữ nguyên cột 'price_raw' để đối chiếu nếu cần
df['price'] = df['price'].apply(parse_price)

print("⏳ Đang chạy chuẩn hóa cột AREA...")
# Tạo cột mới 'area'
df['area'] = df['area'].apply(parse_area)

print("✅ Đã chuẩn hóa xong!")


KẾT QUẢ LOGIC:
Input: 5,5 Triệu/tháng        -> Output: 5,500,000
Input: 9.5 triệu/tháng        -> Output: 9,500,000
Input: 5.000.000 đồng/tháng   -> Output: 5,000,000
Input: 3,5 - 5 Triệu/tháng    -> Output: 4,250,000
Input: 5 đồng/tháng           -> Output: 5
Input: 0,75 Triệu/tháng       -> Output: 750,000
Input: 7 Tỷ                   -> Output: 7,000,000,000

KIỂM TRA AREA:
Input: 30 m2           -> Output: 30.0
Input: 30 - 40 m2      -> Output: 35.0
Input: 4x15            -> Output: 60.0
Input: 5*20 m2         -> Output: 100.0
Input: 15,5 m2         -> Output: 15.5
Input: m2              -> Output: nan
Dữ liệu trước khi chuẩn hóa: (14655, 10)
⏳ Đang chạy chuẩn hóa cột PRICE...
⏳ Đang chạy chuẩn hóa cột AREA...
✅ Đã chuẩn hóa xong!
